# 03 — Model Training

Training and comparison of:

1. Elastic-Net Cox proportional hazards model
2. Random Survival Forest

Patient-level 60/20/20 train/validation/test splitting is used for the synthetic proof of concept.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored

DATA_FILE = Path('../data/processed/synthetic_2000_patients.json')
SEED = 20260916

In [ ]:
with DATA_FILE.open('r', encoding='utf-8') as f:
    patients = json.load(f)

print(f'Patients: {len(patients)}')

## Model-ready data

The following cell expects the synthetic cohort to contain time-to-event information. If the current JSON contains baseline records only, the event-generation/simulation step should be performed in the synthetic-data generation pipeline before model fitting.

In [ ]:
def first_item(value):
    if isinstance(value, list):
        return value[0] if value else {}
    if isinstance(value, dict):
        return value
    return {}

def number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

In [ ]:
rows = []

for patient in patients:
    demo = patient.get('demographics_and_body_size', {})
    echo = first_item(patient.get('echocardiograms', []))

    rows.append({
        'patient_id': patient.get('patient_id'),
        'age': number(demo.get('age')),
        'bmi': number(demo.get('bmi')),
        'mean_gradient': number(echo.get('mean_gradient')),
        'eoa': number(echo.get('eoa')),
        'dvi': number(echo.get('dvi')),
        'lvef': number(echo.get('lvef')),
        'num_echos': len(patient.get('echocardiograms', []))
            if isinstance(patient.get('echocardiograms', []), list) else 0,
        'time': patient.get('time_to_event'),
        'event': patient.get('event'),
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
feature_cols = [
    'age', 'bmi', 'mean_gradient', 'eoa',
    'dvi', 'lvef', 'num_echos'
]

if df['time'].isna().all() or df['event'].isna().all():
    raise ValueError(
        'The JSON does not contain time_to_event/event. '
        'Generate the synthetic survival outcomes before training.'
    )

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(df))

train_end = int(len(df) * 0.60)
val_end = int(len(df) * 0.80)

train_idx = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

X_train = df.loc[train_idx, feature_cols]
X_val = df.loc[val_idx, feature_cols]
X_test = df.loc[test_idx, feature_cols]

y_train = Surv.from_arrays(
    df.loc[train_idx, 'event'].astype(bool),
    df.loc[train_idx, 'time'].astype(float),
)

y_val = Surv.from_arrays(
    df.loc[val_idx, 'event'].astype(bool),
    df.loc[val_idx, 'time'].astype(float),
)

y_test = Surv.from_arrays(
    df.loc[test_idx, 'event'].astype(bool),
    df.loc[test_idx, 'time'].astype(float),
)

In [ ]:
imputer = SimpleImputer(strategy='median', keep_empty_features=True)
X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)
X_test = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## Elastic-Net Cox

In [ ]:
cox = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    alpha_min_ratio=0.01,
    n_alphas=100,
    max_iter=100000,
)

cox.fit(X_train_scaled, y_train)

best_alpha = None
best_score = -np.inf

for alpha in cox.alphas_:
    pred = cox.predict(X_val_scaled, alpha=alpha)
    score = concordance_index_censored(
        y_val['event'], y_val['time'], pred
    )[0]
    if score > best_score:
        best_score = score
        best_alpha = alpha

cox_test_pred = cox.predict(X_test_scaled, alpha=best_alpha)
cox_test_cindex = concordance_index_censored(
    y_test['event'], y_test['time'], cox_test_pred
)[0]

print('Best alpha:', best_alpha)
print('Validation C-index:', best_score)
print('Test C-index:', cox_test_cindex)

## Random Survival Forest

In [ ]:
rsf = RandomSurvivalForest(
    n_estimators=500,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    n_jobs=-1,
    random_state=SEED,
)

rsf.fit(X_train, y_train)

rsf_val_pred = rsf.predict(X_val)
rsf_test_pred = rsf.predict(X_test)

rsf_val_cindex = concordance_index_censored(
    y_val['event'], y_val['time'], rsf_val_pred
)[0]

rsf_test_cindex = concordance_index_censored(
    y_test['event'], y_test['time'], rsf_test_pred
)[0]

print('Validation C-index:', rsf_val_cindex)
print('Test C-index:', rsf_test_cindex)